In [ ]:
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
import numpy as np

In [6]:
# Generate heatmaps of clusters correlations for each decade
for decade in np.arange(1900, 2010, 10):
    
    df = pd.read_csv(f'src/data/heatmaps_data/heatmap_data_{decade}.csv', index_col=0)

    # Extract x-axis and y-axis labels
    x_labels = df.columns.tolist()
    y_labels = df.index.tolist()

    # Create the heatmap
    fig = go.Figure(data=go.Heatmap(
        z=df.values,  
        x=x_labels,   
        y=y_labels,   
        colorscale='YlOrRd', 
    ))

    # Add title and labels
    fig.update_layout(
    title= f'Correlation between history and movies summaries clusters for {decade}s',
    title_x=0.5,  
    title_font=dict(size=18),  
    xaxis_title="History",  
    yaxis_title="Movies" 
    )

    fig.show()
    

In [7]:
# Create a CSV file with correlations > 0.35

higher_correlations = []
for decade in np.arange(1900, 2010, 10):

    df = pd.read_csv(f'src/data/heatmaps_data/heatmap_data_{decade}.csv', index_col=0)

    # Convert all columns to numeric, forcing errors to NaN 
    df = df.apply(pd.to_numeric, errors='coerce')

    # Use stack() to flatten the dataframe and filter correlations greater than 0.35
    filtered_df = df.stack()
    filtered_df = filtered_df[filtered_df > 0.35]

    # Loop through the values and add them to the higher_accuracy list
    for (row_label, col_label), correlation_value in filtered_df.items():
        higher_correlations.append({
            'Decade': decade,
            'Movies cluster': row_label,
            'History cluster': col_label,
            'Correlation': correlation_value
        })

# Create a dataframe from the higher_accuracy list
higher_correlations_df = pd.DataFrame(higher_correlations)

# Save the result 
higher_correlations_df.to_csv('src/data/higher_correlations_data.csv', index=False)


In [4]:
# Find the max similarities for each decade
max_similarities = []

for decade in np.arange(1900, 2010, 10):
   
    df = pd.read_csv(f'src/data/max_similarity_plots_data/bar_plot_data_{decade}.csv', index_col=0)

    df = df.apply(pd.to_numeric, errors='coerce')

    # Find the maximum value in the dataframe
    max_value = df.max().max()

    # Row and column labels for the maximum value
    row_label, col_label = np.unravel_index(df.values.argmax(), df.shape)

    # Add the result to the list
    max_similarities.append({
        'Decade': decade,
        'History cluster': df.index[row_label],
        'Similarity': max_value
    })

# Create a dataframe from the max_similarities list
max_similarities_df = pd.DataFrame(max_similarities)

# Save the result 
max_similarities_df.to_csv('src/data/max_similarities_data.csv', index=False)


In [5]:
# Timeline of max similarities

df = pd.read_csv('src/data/max_similarities_data.csv')

df['Cluster'] = df['History cluster'].astype(str)  

# Create the bar plot
fig = px.bar(df, x='Decade', y='Similarity',
             labels={'Similarity': 'Max Corr in Movies Summaries'}, height=400)

# Add cluster names on top of each bar
fig.update_traces(text=df['Cluster'], textposition='outside', texttemplate='%{text}', textfont_size=10, textfont_weight='bold')

colors = ['#ba3c3c', '#dc6c3a']  
fig.update_traces(marker_color=[colors[i % len(colors)] for i in range(len(df))])

# Hide the legend
fig.update_layout(showlegend=False)

fig.show()

In [ ]:
# Create a bubble chart of higher correlations

df = pd.read_csv('src/data/higher_correlations_data.csv')

# Ensure 'Decade' is treated as a categorical variable by converting it to string
df['Decade'] = df['Decade'].astype(str)

# Normalize the Correlation column to a range between 0 and 1 to see the differences better
df['Normalized Correlation'] = (df['Correlation'] - df['Correlation'].min()) / (df['Correlation'].max() - df['Correlation'].min())

# Create the bubble chart
fig = px.scatter(
    df,
    x='Movies cluster',
    y='History cluster',
    size='Normalized Correlation',   
    size_max=40,                     
    color='Decade',                  
    color_discrete_sequence=px.colors.sequential.Turbo_r,  
    height=400,
    labels={
        "Movies cluster": "Movies Clusters",
        "History cluster": "History Clusters",
        "Correlation": "Correlation Coefficient",
        "Decade": "Decade"
    }
)

# Update layout to add a title and set the legend properties
fig.update_layout(
    legend_title_text='Decade',  # Set the title of the legend
    legend=dict(
        title_font=dict(size=14, color="black"),
        font=dict(size=12),
        bgcolor="rgba(255, 255, 255, 0.7)",  # Semi-transparent background
        bordercolor="black",
        borderwidth=1,
    ),
    title={
        'text': 'Higher Correlations between History and Movies Summaries Clusters',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Movies Clusters',
    yaxis_title='History Clusters',
)

fig.show()
